# 13 Unified Observed/Expected Framework

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
NOTEBOOK_OUTPUT_DIR = PROJECT_ROOT / 'outputs' / '13_unified_expected_observed_framework'
FIGURE_DIR = NOTEBOOK_OUTPUT_DIR / 'figures'
TABLE_DIR = NOTEBOOK_OUTPUT_DIR / 'tables'
DATA_OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR / 'data'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GENDER_CATEGORIES = ['MM', 'MW', 'WM', 'WW']
pd.set_option('display.max_columns', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

## Standardize Baseline Outputs

In [2]:
def normalize_result(df, fallback_baseline=None, fallback_slice=None):
    x = df.copy()
    rename = {'obs_exp_ratio': 'oe_ratio', 'difference_pct_of_expected': 'pct_diff_from_expected', 'observed': 'observed_citations', 'expected': 'expected_citations', 'gender_category': 'gender_cat'}
    x = x.rename(columns={k: v for k, v in rename.items() if k in x.columns and v not in x.columns})
    if 'gender_cat' not in x.columns and x.index.name in {'first_last_cat', 'gender_cat', None}:
        if set(map(str, x.index)).intersection(GENDER_CATEGORIES):
            x = x.reset_index().rename(columns={x.reset_index().columns[0]: 'gender_cat'})
    if 'baseline' not in x.columns:
        x['baseline'] = fallback_baseline
    if 'citation_slice' not in x.columns:
        x['citation_slice'] = fallback_slice
    if 'pct_diff_from_expected' not in x.columns and 'oe_ratio' in x.columns:
        x['pct_diff_from_expected'] = (x['oe_ratio'] - 1) * 100
    keep = [c for c in ['baseline', 'citation_slice', 'gender_cat', 'observed_citations', 'expected_citations', 'oe_ratio', 'pct_diff_from_expected', 'n_papers', 'coverage_share', 'calibration_factor', 'n_edges'] if c in x.columns]
    return x[keep].copy()
pieces = []

## Load Baseline Outputs

In [3]:
d0_all = normalize_result(pd.read_csv(PROJECT_ROOT / 'outputs' / '06_baseline_d0_global_paper_share' / 'tables' / 'baseline_d0_global_all_to_cs.csv', index_col=0), 'D0_global', 'All→CS')
d0_cs = normalize_result(pd.read_csv(PROJECT_ROOT / 'outputs' / '06_baseline_d0_global_paper_share' / 'tables' / 'baseline_d0_global_cs_to_cs.csv', index_col=0), 'D0_global', 'CS→CS')
pieces.extend([d0_all, d0_cs])
baseline_files = [PROJECT_ROOT / 'outputs' / '07_baseline_d1_decade' / 'tables' / 'd1_decade_oe_by_gender.csv', PROJECT_ROOT / 'outputs' / '08_baseline_d2_subfield_topic' / 'tables' / 'd2_decade_subfield_oe_by_gender.csv', PROJECT_ROOT / 'outputs' / '09_baseline_d3_venue' / 'tables' / 'd3_decade_venue_oe_by_gender.csv', PROJECT_ROOT / 'outputs' / '10_baseline_r1_taxonomy_relevance' / 'tables' / 'r1_taxonomy_relevance_oe_by_gender.csv', PROJECT_ROOT / 'outputs' / '11_baseline_n1_bibliographic_coupling' / 'tables' / 'n1_bibliographic_coupling_oe_by_gender.csv', PROJECT_ROOT / 'outputs' / '12_baseline_n2_cocitation' / 'tables' / 'n2_cocitation_oe_by_gender.csv']
for path in baseline_files:
    pieces.append(normalize_result(pd.read_csv(path)))
unified = pd.concat(pieces, ignore_index=True, sort=False)
unified['gender_cat'] = pd.Categorical(unified['gender_cat'], categories=GENDER_CATEGORIES, ordered=True)
unified = unified.sort_values(['citation_slice', 'baseline', 'gender_cat']).reset_index(drop=True)
display(unified)

,baseline,citation_slice,gender_cat,observed_citations,expected_citations,oe_ratio,pct_diff_from_expected,n_papers,n_edges,coverage_share,calibration_factor
0,D0_global,All→CS,MM,"6,942,653.0000","6,350,769.0186",1.0932,9.3199,"790,617.0000",NaN,NaN,NaN
1,D0_global,All→CS,MW,"641,435.0000","782,374.4640",0.8199,-18.0143,"97,399.0000",NaN,NaN,NaN
2,D0_global,All→CS,WM,"812,529.0000","1,106,549.1090",0.7343,-26.5709,"137,756.0000",NaN,NaN,NaN
3,D0_global,All→CS,WW,"289,620.0000","446,544.4084",0.6486,-35.1419,"55,591.0000",NaN,NaN,NaN
4,D1_decade,All→CS,MM,"6,942,653.0000","6,666,725.6879",1.0414,4.1389,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
83,R1_same_topic,CS→CS,WW,"124,564.0000","169,177.0063",0.7363,-26.3706,NaN,"3,630,264.0000",NaN,NaN
84,R1_taxonomy_weighted,CS→CS,MM,"4,502,409.7500","4,348,954.6237",1.0353,3.5286,NaN,"7,023,500.0000",NaN,NaN
85,R1_taxonomy_weighted,CS→CS,MW,"435,854.7500","451,746.5448",0.9648,-3.5179,NaN,"7,023,500.0000",NaN,NaN
86,R1_taxonomy_weighted,CS→CS,WM,"549,627.7500","611,389.4243",0.8990,-10.1019,NaN,"7,023,500.0000",NaN,NaN


## O/E Comparison

In [4]:
pivot_all = unified[unified['citation_slice'].eq('All→CS')].pivot_table(index='gender_cat', columns='baseline', values='oe_ratio', aggfunc='first')
pivot_cs = unified[unified['citation_slice'].eq('CS→CS')].pivot_table(index='gender_cat', columns='baseline', values='oe_ratio', aggfunc='first')
pivot_all

baseline,D0_global,D1_decade,D2_decade_subfield,D3_decade_venue,N1_bibliographic_coupling,N2_cocitation,R1_cross_subfield,R1_cross_topic_same_subfield,R1_same_subfield,R1_same_topic,R1_taxonomy_weighted
gender_cat,,,,,,,,,,,
MM,1.0932,1.0414,1.0414,1.0309,1.0500,1.0115,1.0535,1.0480,1.0324,1.0263,1.0371
MW,0.8199,0.9382,0.9354,0.9487,0.8806,0.9847,0.8852,0.9069,0.9705,0.9940,0.9577
WM,0.7343,0.8841,0.8765,0.9071,0.8273,0.9188,0.8284,0.8425,0.9097,0.9343,0.8932
WW,0.6486,0.6950,0.7123,0.7619,0.7988,0.9674,0.6927,0.7019,0.7271,0.7363,0.7067


## Save Outputs

In [5]:
unified.to_csv(TABLE_DIR / 'unified_observed_expected_all_baselines.csv', index=False)
pivot_all.to_csv(TABLE_DIR / 'unified_oe_pivot_all_to_cs.csv')
pivot_cs.to_csv(TABLE_DIR / 'unified_oe_pivot_cs_to_cs.csv')
print('Saved:', TABLE_DIR)

Saved: c:\Users\blaze\OneDrive\Documents\GitHub\citation-gender-imbalance\outputs\13_unified_expected_observed_framework\tables
